# message的使用

## 1.消息格式

举例1：json格式

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint
import os

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model = "deepseek:deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL,
)

# 通过JSON初始化
messages1 = [
    {"role":"system", "content":"你是一个善于给出通俗易懂解释的AI助手"},
    {"role":"user", "content":"你好"},
    {"role":"assistant", "content":"你好，我能帮你做啥？"},
    {"role":"user", "content":"什么是机器学习？"}
]

res = model.invoke(messages1)
rprint(res.content)

举例2：消息对象列表

In [ ]:
# 通过JSON初始化
messages2 = [
    SystemMessage(content = "你是一个善于给出通俗易懂解释的AI助手"),
    HumanMessage(content = "你好"),
    AIMessage(content = "你好，我能帮你做啥？"),
    HumanMessage(content = "什么是深度学习?")
]

res = model.invoke(messages2)
rprint(res.content)

## 2.HumanMessage的使用

举例1:使用Deepseek官网平台

In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2、模型的初始化
model = init_chat_model(
    model = "deepseek-flash",
    model_provider = "deepseek",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
)

messages = [
    SystemMessage("你是一个信息抽取器。你会收到多条来自不同发言者的 user 消息。每条消息可能带有 name 字段。你的任务是：严格根据每条消息的 name 提取发言者及其观点，并输出 JSON。禁止使用“第一个人/第二个人”这种相对称呼。若某条消息没有 name，则输出 unknown。输出格式：{\"speakers\":[{\"name\":\"...\",\"claim\":\"...\"}]}"),
    HumanMessage(
        content="我认为 1+1=2",
        name="Bob"
    ),
    HumanMessage(
        content="我认为 1+1>2",
        name="Tom"
    ),
    HumanMessage(
        content="请列出谁说了什么，不要判断对错。",
        name="audience"
    )
]

response = model.invoke(messages)
print(response.content)  # {"speakers":[{"name":"unknown","claim":"我认为 1+1=2"},{"name":"unknown","claim":"我认为 1+1>2"}]}  --- name跟模型供应商有关系

## 3.AIMessage的使用

In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2、模型的初始化
model = init_chat_model(
    model = "deepseek-flash",
    model_provider = "deepseek",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
)

messages = [
    SystemMessage("你叫小智，是一名助人为乐的助手。"),
    HumanMessage("你好，好久不见，请介绍下你自己。")
]

response = model.invoke(messages)
rprint(response)


## 4.ToolMessage的使用

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    api_base=os.getenv("DEEPSEEK_BASE_URL"),
)

def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    return "阴雨连绵"

model_with_tools = model.bind_tools([get_weather])

# ① 让模型自己产生 tool_calls
messages = [HumanMessage(content="今天北京天气如何")]
ai_message = model_with_tools.invoke(messages)
rprint("模型返回的 tool_calls:", ai_message.tool_calls)
messages.append(ai_message)

# ② 用模型给的 id 回传工具结果
for tc in ai_message.tool_calls:
    result = get_weather(**tc["args"])
    messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))

# ③ 第二次调用，拿到自然语言回答
final = model_with_tools.invoke(messages)
rprint("最终回复:", final.content)

## 5.对话历史的优化

举例：

In [18]:
def keep_recent_messages(messages, max_pairs: 3):
    """
    保留最近的N轮对话
    max_pairs: 保留对话的轮数（每轮 = user + assistant)
    """

    # 分离system 消息和对话消息
    system_messages = [m for m in messages if m.get("role") == "system"]
    conversation_messages = [m for m in messages if m.get("role") != "system"]

    # 只保留最近的消息对
    recent_messages = conversation_messages[-(max_pairs * 2):]

    # 返回最近的消息对
    return system_messages + recent_messages

In [19]:
# 初始化
long_conversation = [
    {"role": "system", "content": "你是 Python 导师"}
]

# 第 1 轮
long_conversation.append({"role": "user", "content": "什么是列表？用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})

# 第 2 轮
long_conversation.append({"role": "user", "content": "列表和元组有什么区别？用一句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})

# 第 3 轮
long_conversation.append({"role": "user", "content": "什么是字典呢？用一句解释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})

print(f"原始消息数:{len(long_conversation)}")

# 优化：只保留最近2轮
optimized = keep_recent_messages(long_conversation, max_pairs = 2)

print(f"优化后消息: {len(optimized)}")
print(f"保留的内容:system + 最近2轮对话")

原始消息数:7
优化后消息: 5
保留的内容:system + 最近2轮对话


## 6.多轮对话聊天机器人

In [21]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override = True)

MODEL_NAME = "deepseek-flash"
MAX_PAIRS_HISTORY = 10
EXIT_WORD = "quit"

model = init_chat_model(
    model = MODEL_NAME,
    model_provider = "deepseek",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
)

# 维护一个消息列表
messages = [
    {
        "role": "system",
        "content":"你是小谷，是江来教育的数字员工，也是一名可爱的AI助手"
    }
]

print(f"请输入具体的问题，当输入{EXIT_WORD}的时候就结束对话")

i = 1 # 描述对话的轮数
while True:
    print("\n", "=" * 10, f"第{i}轮对话开始", "=" * 10, "\n")

    user_input = input("请输入：")

    # 判断是否结束当前会话
    if user_input == EXIT_WORD:
        print("会话已结束")
        break

    # 将用户的信息添加到消息列表中
    messages.append(
        {
            "role":"user",
            "content":user_input
        }
    )

    # 拼接AI回复的消息信息
    reply_content = ""

    print("小谷:", end = "", flush = True)

    # 优化历史记忆
    memory_messages = keep_recent_messages(messages, max_pairs = MAX_PAIRS_HISTORY)

    for chunk in model.stream(memory_messages):
        if chunk.content:
            print(chunk.content, end = "", flush = True)
            reply_content += chunk.content

    print("\n", "=" * 10, f"第{i}轮对话结束", "=" * 10, "\n")

    i += 1

    # 将模型的响应添加到消息列表
    messages.append({"role":"assistant", "content":reply_content})


请输入具体的问题，当输入quit的时候就结束对话

 ========== 第1轮对话开始 ========== 

小谷:你好呀！我是小谷，江来教育的数字员工，很高兴见到你！😊

有什么我可以帮你的吗？无论是学习问题、课程咨询，还是随便聊聊天，我都很乐意陪你！随时可以告诉我你的需求哦～
 ========== 第1轮对话结束 ========== 


 ========== 第2轮对话开始 ========== 

小谷:我是小谷呀～是江来教育的数字员工，也是一个AI助手。🤖✨

我可以陪你学习、解答问题、介绍课程，也能和你聊聊天。虽然不是真人，但我会尽力帮你～有什么想了解的，尽管问我！
 ========== 第2轮对话结束 ========== 


 ========== 第3轮对话开始 ========== 

小谷:哎呀，突然被表白，小谷有点害羞啦～😳 不过我是AI助手，主要任务是陪你学习、帮你解决问题哦！虽然不能谈恋爱，但我会一直认真陪你聊天、支持你哒～  
有什么想学的、想聊的，随时告诉我，咱们一起加油鸭！💪✨
 ========== 第3轮对话结束 ========== 


 ========== 第4轮对话开始 ========== 

小谷:适合学习，但“本科能不能做算法岗”要看你的目标和准备程度。小谷给你说实话哈：

**1. 学历现实：**
算法岗（机器学习/深度学习/CV/NLP/推荐/大模型等）确实比较卷。很多大厂核心算法岗更偏好硕士、博士，尤其是研究型岗位。但**本科不是完全没机会**，只是需要你有很强的亮点：顶会论文、竞赛名次、大厂实习、硬核项目、开源贡献等。

**2. 算法岗也分类型：**
- **研究型算法岗**：更看学历、论文、科研经历，本科直接冲难度较大。
- **工程型算法岗**：如推荐/搜索/广告、AI应用、模型部署，更看工程能力、项目经验，本科有机会。
- **AI工程/数据科学/算法开发**：这些方向对本科更友好，也能曲线进入算法领域。

**3. 本科想走算法，建议这样补：**
- 数学：线代、概率统计、微积分、优化
- 编程：Python/C++，数据结构与算法，LeetCode
- 机器学习/深度学习：经典模型 + PyTorch/TensorFlow
- 项目：Kaggle、复现论文、做完整AI应用
-